# TrafficEye: time budget on a Tesla T4

Runtime → Change runtime type → **T4 GPU**. Then run the cells top to bottom.
The notebook clones the public repository, installs the pinned requirements, runs the organizers' harness on a sample clip and prints Part A / Part B seconds against the 3× budget.

Put a sample clip (for example `C3905.MP4`) in your Google Drive folder `wiut-samples/`, or let the last cell generate a synthetic 4K clip if you have none.

In [ ]:
!nvidia-smi -L
!git clone --depth 1 https://github.com/TERMINATOOOOOOOr/Zadaniye.git wiut-cv
%cd wiut-cv
!pip install -q -r requirements.txt
import torch, ultralytics, cv2; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'ultralytics', ultralytics.__version__, 'cv2', cv2.__version__)

In [ ]:
# Option A: a real sample from Google Drive (folder wiut-samples/)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/videos && ls -la /content/drive/MyDrive/wiut-samples/ && cp /content/drive/MyDrive/wiut-samples/*.MP4 /content/videos/ 2>/dev/null; ls -la /content/videos

In [ ]:
# Option B (no sample at hand): a synthetic 4K 30 fps clip of 60 s; decoding cost is realistic, the scene is not
import os
if not any(f.lower().endswith('.mp4') for f in os.listdir('/content/videos')):
    !python tools/ci_smoke.py make-video /content/videos/synthetic4k.mp4 --seconds 60 --fps 30 --width 3840 --height 2160
!ls -la /content/videos

In [ ]:
!python run_submission.py --videos /content/videos --out /content/pred_t4.json --team "Air MAX"
import json
d = json.load(open('/content/pred_t4.json'))
print('| Footage | Length | Part A | Part B | Total | Limit |')
print('|---|---|---|---|---|---|')
for name, l in d['log'].items():
    dur = l['duration']
    print(f"| {name} | {dur:.0f} s | {l['part_a_sec']:.0f} s ({l['part_a_sec']/dur:.2f}×) | {l['part_b_sec']:.0f} s ({l['part_b_sec']/dur:.2f}×) | {l['total_sec']:.0f} s = {l['total_sec']/dur:.2f}× | 3× |")
    print('  events:', len(d['videos'][name]['events']), 'errors:', l['errors'])

In [ ]:
# Determinism on the T4: run once more and compare
!python run_submission.py --videos /content/videos --out /content/pred_t4_b.json --team "Air MAX" > /dev/null
import json
a = json.load(open('/content/pred_t4.json'))['videos']; b = json.load(open('/content/pred_t4_b.json'))['videos']
print('identical events and risk:', all(a[k]['events'] == b[k]['events'] and a[k]['risk'] == b[k]['risk'] for k in a))